# Toothwise — Train YOLOv8 + YOLO12 dental detectors

Runs in Google Colab. Before starting: **Runtime → Change runtime type → T4 GPU** (or better).

This notebook trains **both** models the backend expects (dual-model ensemble), then
packages them as `yolov8_dental.pt` and `yolov12_dental.pt` for you to drop into
`backend/models_weights/`.

**Labels the system uses (5 canonical classes — matches `backend/app/ml/labels.py`):**

| # | Label | Severity |
|---|-------|----------|
| 0 | `healthy_tooth` | 0 (baseline) |
| 1 | `cavity` | 70 (high) |
| 2 | `gum_inflammation` | 45 (moderate-high) |
| 3 | `plaque` | 25 (low-moderate) |
| 4 | `tartar` | 35 (moderate) |

> Your dataset class names do **not** have to exactly match the list above. The backend
> automatically normalizes common synonyms (`caries→cavity`, `calculus→tartar`,
> `gingivitis→gum_inflammation`, etc.) via `LABEL_ALIASES` in `labels.py`. Check that
> file if you use non-standard names.


## 1. Install dependencies

In [ ]:
# ultralytics ≥ 8.3 ships YOLOv12 support out of the box — no separate package needed.
!pip install -q 'ultralytics>=8.3' roboflow

## 1b. Confirm you're on a GPU

Do this **before** touching the dataset cells. If this errors or shows no GPU,
go to **Runtime → Change runtime type** and pick a GPU (T4 is the free-tier
default; A100/L4 if you have Colab Pro/Pro+), then re-run from the top.
Training silently on CPU is the #1 cause of a Colab run that looks "stuck".

In [ ]:
!nvidia-smi

## 2. Get your dataset

**Option A — Roboflow (recommended):** paste the download snippet Roboflow gives you
after exporting your project in **YOLOv8** format. It looks like the cell below —
replace the placeholders with your own project details.


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)  # your version number
dataset = version.download("yolov8")

print(dataset.location)

**Option B — manual zip upload:** if you annotated locally (LabelImg/CVAT) and
already have a YOLO-format folder (`train/`, `valid/`, `data.yaml`), zip it and
run this cell instead of the Roboflow one above.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # select your dataset .zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/dental_dataset")

dataset_location = "/content/dental_dataset"
print(os.listdir(dataset_location))

## 3. Sanity-check your data.yaml before spending GPU time

In [ ]:
import glob

data_yaml_path = glob.glob(f"{dataset.location}/data.yaml") if "dataset" in globals() else glob.glob(f"{dataset_location}/data.yaml")
data_yaml_path = data_yaml_path[0]
print(data_yaml_path)
print(open(data_yaml_path).read())

## 3b. (Optional) Patch data.yaml to match the system's canonical class names

Run this cell only if your `data.yaml` uses different class names (e.g. `caries` instead
of `cavity`). This rewrites the `names` block so the exported `.pt` files emit the exact
canonical labels the backend expects — you won't have to rely on `LABEL_ALIASES` at all.

**Skip this cell if your dataset is already labeled with the 5 canonical names.**

In [ ]:
import yaml

# Edit this map: {your_dataset_name: canonical_name}
RENAME_MAP = {
    "caries": "cavity",
    "decay": "cavity",
    "calculus": "tartar",
    "gingivitis": "gum_inflammation",
    "healthy": "healthy_tooth",
    "normal": "healthy_tooth",
}

with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)

original_names = cfg.get("names", [])
cfg["names"] = [RENAME_MAP.get(n.lower(), n) for n in original_names]

with open(data_yaml_path, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print("Before:", original_names)
print("After: ", cfg["names"])

## 4. Train YOLOv8s

100 epochs is a reasonable capstone default with early stopping (`patience=20`) so it
won't run the full 100 if it stops improving. Increase `epochs` if you have a large,
balanced dataset and Colab time to spare.

`batch=-1` lets Ultralytics auto-size the batch to your GPU's free memory instead of
guessing a fixed number, and `cache=True` caches decoded images in RAM so epochs after
the first don't re-read/re-decode every image from disk - together these are usually the
biggest no-accuracy-cost speedup available. If you hit a CUDA out-of-memory error, replace
`batch=-1` with a fixed number like `batch=16`.

> **`yolov8s.pt` vs `yolov8n.pt`**: `s` (small) typically achieves 5–10% higher mAP than `n` (nano)
> on a medical dataset with minimal extra training time on a T4. Use `n` only if you hit OOM errors.

In [ ]:
from ultralytics import YOLO

model_v8 = YOLO("yolov8s.pt")  # pretrained COCO weights as starting point
results_v8 = model_v8.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=-1,       # auto-select the largest batch that fits in GPU memory
    cache=True,     # cache decoded images in RAM - big speedup after epoch 1
    patience=20,
    name="toothwise_yolov8",
)

## 5. Train YOLO12s

Same dataset, same hyperparameters, so the two models are directly comparable for
your ensemble and for your results/methodology chapter.

> YOLOv12 requires `ultralytics>=8.3`. If the cell below raises `Model not found`,
> run `!pip install -q -U ultralytics` and restart the runtime.

In [ ]:
model_v12 = YOLO("yolo12s.pt")  # pretrained COCO weights as starting point
results_v12 = model_v12.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=-1,       # auto-select the largest batch that fits in GPU memory
    cache=True,     # cache decoded images in RAM - big speedup after epoch 1
    patience=20,
    name="toothwise_yolo12",
)

## 6. Check results — confusion matrices + metrics

In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/runs/detect/toothwise_yolov8/confusion_matrix.png"))
display(Image(filename="/content/runs/detect/toothwise_yolo12/confusion_matrix.png"))

In [ ]:
metrics_v8 = model_v8.val()
metrics_v12 = model_v12.val()

print("YOLOv8  mAP50-95:", metrics_v8.box.map)
print("YOLO12  mAP50-95:", metrics_v12.box.map)
print("\nPer-class mAP50 (YOLOv8):", metrics_v8.box.maps)
print("Per-class mAP50 (YOLO12): ", metrics_v12.box.maps)

## 7. Verify the exact class names your models will emit

Cross-check this against `LABELS` / `LABEL_ALIASES` in
`backend/app/ml/labels.py`. Anything not covered there gets passed through
as lowercased raw text instead of silently dropped, so mismatches are
easy to spot in the app's detection results.

In [ ]:
CANONICAL = ["healthy_tooth", "cavity", "gum_inflammation", "plaque", "tartar"]

v8_names  = list(model_v8.names.values())
v12_names = list(model_v12.names.values())

print("YOLOv8 classes :", v8_names)
print("YOLO12 classes :", v12_names)

# Warn about any name the backend won't auto-normalize
ALIASES = {"healthy", "health", "normal", "caries", "decay", "tooth-decay",
           "cavities", "gingivitis", "gingiva", "gum disease", "calculus", "tartar buildup"}
for name in set(v8_names + v12_names):
    if name not in CANONICAL and name.lower() not in ALIASES:
        print(f"  ⚠  '{name}' is not in LABELS or LABEL_ALIASES — add it to labels.py")

## 8. Package weights for the backend

Renames both `best.pt` checkpoints to what `backend/.env` expects
(`yolov8_dental.pt`, `yolov12_dental.pt`) and downloads them.

In [ ]:
import shutil
from google.colab import files

shutil.copy("/content/runs/detect/toothwise_yolov8/weights/best.pt", "/content/yolov8_dental.pt")
shutil.copy("/content/runs/detect/toothwise_yolo12/weights/best.pt", "/content/yolov12_dental.pt")

files.download("/content/yolov8_dental.pt")
files.download("/content/yolov12_dental.pt")

## 9. Wire it into the backend

1. Move both downloaded files into `backend/models_weights/`.
2. In `backend/.env`, set `USE_MOCK_MODEL=false`.
3. **Delete the old KNN artifact** so it retrains on first boot against your real model's outputs:
   ```bash
   del backend\app\ml\artifacts\knn_recommender.joblib
   ```
   Or retrain it explicitly:
   ```bash
   cd backend
   .venv\Scripts\python.exe -m app.ml.train_knn
   ```
4. Restart the backend (`docker compose restart backend`, or re-run
   `uvicorn` if running natively).
5. Upload a real screening photo and confirm the detections + prevention
   plan look reasonable. If the model is undertrained (small/imbalanced
   dataset — common for a first capstone pass), keep `USE_MOCK_MODEL=true`
   for live demos and present these training metrics + confusion matrices
   as evidence of the pipeline working, with real-model results as a
   secondary appendix. That's a completely normal and defensible capstone
   framing.
